# Binaries — documentation examples

Companion notebook to the **Binaries** documentation page
(`docs/source/binaries.rst`). One section per code snippet.

The **native Keplerian** sections run with the core install; the **PHOEBE**
sections require `pip install "stellar-spice[phoebe]"` and are shipped
unexecuted.

## Native Keplerian binary

Two blackbody stars on a Keplerian orbit — runs with the core install only.
The components must be cast to the line of sight with `get_mesh_view` so
mutual occlusions can be resolved.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import numpy as np
import jax.numpy as jnp
from spice.models import IcosphereModel, Binary
from spice.models.binary import add_orbit, evaluate_orbit_at_times
from spice.models.mesh_view import get_mesh_view
from spice.spectrum import simulate_observed_flux, Blackbody, AB_passband_luminosity
from spice.spectrum.filter import GaiaG

bb = Blackbody()
los = jnp.array([0.0, 1.0, 0.0])

primary = get_mesh_view(
    IcosphereModel.construct(1000, 1.0, 1.0, bb.solar_parameters, bb.parameter_names), los)
secondary = get_mesh_view(
    IcosphereModel.construct(1000, 0.8, 0.8, bb.solar_parameters, bb.parameter_names), los)

binary = Binary.from_bodies(primary, secondary)

binary = add_orbit(
    binary,
    P=1.0,                    # orbital period [years]
    ecc=0.1,                  # eccentricity
    T=0.0,                    # time of periastron passage [years]
    i=np.pi / 3,              # inclination [rad]
    omega=0.0,                # argument of periastron [rad]
    Omega=0.0,                # longitude of the ascending node [rad]
    mean_anomaly=0.0,         # mean anomaly at the reference time [rad]
    reference_time=0.0,       # reference time [years]
    vgamma=0.0,               # systemic velocity [km/s]
    orbit_resolution_points=50,
)

### Evaluate the orbit and build a light curve

In [ ]:
times = jnp.linspace(0.0, 1.0, 20)
primaries, secondaries = evaluate_orbit_at_times(binary, times)

wavelengths = np.linspace(900, 40000, 500)
gaia_g = GaiaG()
light_curve = [
    AB_passband_luminosity(
        gaia_g,
        wavelengths,
        simulate_observed_flux(bb.intensity, p1, np.log10(wavelengths))[:, 0]
        + simulate_observed_flux(bb.intensity, p2, np.log10(wavelengths))[:, 0],
    )
    for p1, p2 in zip(primaries, secondaries)
]

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.asarray(times), np.asarray(light_curve), "o-")
ax.invert_yaxis()  # magnitudes: smaller = brighter
ax.set_xlabel("Time [yr]")
ax.set_ylabel("Gaia G [AB mag]")
plt.show()

## PHOEBE Configuration and Binary System Setup

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import phoebe
import numpy as np
from phoebe.parameters.dataset import _mesh_columns

# Create a default binary system
b = phoebe.default_binary()

# Define time points (in days)
times = np.linspace(0, 1, 100)

# Add datasets
b.add_dataset('mesh', compute_times=times, columns=_mesh_columns, dataset='mesh01')
b.add_dataset('orb', compute_times=times, dataset='orb01')
b.add_dataset('lc', compute_times=times, passband='Johnson:V', dataset='lc01')

b.set_value('distance@system', 10)  # in solar radii

# Make sure to set the coordinates to 'uvw'
b.run_compute(coordinates='uvw')

## Wrapping the bundle for SPICE and synthesising a spectrum

In [ ]:
import numpy as np
import jax.numpy as jnp
from spice.models import PhoebeBinary
from spice.models.binary import evaluate_orbit
from spice.models.phoebe_utils import PhoebeConfig
from spice.spectrum import simulate_observed_flux, Blackbody

bb = Blackbody()

# Wrap the PHOEBE meshes (read-only inside SPICE)
config = PhoebeConfig(b, 'mesh01')
binary = PhoebeBinary.construct(config, ['teff', 'logg', 'abun'])

# Evaluate the components at a snapshot time, then synthesise a spectrum
primary, secondary = evaluate_orbit(binary, config.times[0])
wavelengths = np.logspace(3, 4, 1000)
flux = simulate_observed_flux(bb.intensity, primary, np.log10(wavelengths))